## 1) Import libraries and data 

In [1]:
# Import libraries 

import sqlite3
import pandas as pd 

In [147]:
pip install yagmail 

Note: you may need to restart the kernel to use updated packages.


In [203]:
# Import CSV file 

donors = pd.read_csv('C:/Users/lg2021/OneDrive/Job/Data/Kaggle/Operations/Fundraising data/Fundraising_donor_data.csv')
contacts = pd.read_csv('C:/Users/lg2021/OneDrive/Job/Data/Kaggle/Operations/Fundraising data/Fundraising_contact_reports.csv')

In [204]:
pd.set_option('display.max_columns', None)

In [205]:
donors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34508 entries, 0 to 34507
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   34508 non-null  int64  
 1   ZIPCODE              34417 non-null  float64
 2   AGE                  13318 non-null  float64
 3   MARITAL_STATUS       9940 non-null   object 
 4   GENDER               34015 non-null  object 
 5   MEMBERSHIP_IND       34508 non-null  object 
 6   ALUMNUS_IND          34508 non-null  object 
 7   PARENT_IND           34508 non-null  object 
 8   HAS_INVOLVEMENT_IND  34508 non-null  object 
 9   WEALTH_RATING        2709 non-null   object 
 10  DEGREE_LEVEL         7606 non-null   object 
 11  PREF_ADDRESS_TYPE    30465 non-null  object 
 12  EMAIL_PRESENT_IND    34508 non-null  object 
 13  CON_YEARS            34508 non-null  int64  
 14  PrevFYGiving         34508 non-null  object 
 15  PrevFY1Giving        34508 non-null 

In [206]:
donors.head()

,ID,ZIPCODE,AGE,MARITAL_STATUS,GENDER,MEMBERSHIP_IND,ALUMNUS_IND,PARENT_IND,HAS_INVOLVEMENT_IND,WEALTH_RATING,DEGREE_LEVEL,PREF_ADDRESS_TYPE,EMAIL_PRESENT_IND,CON_YEARS,PrevFYGiving,PrevFY1Giving,PrevFY2Giving,PrevFY3Giving,PrevFY4Giving,CurrFYGiving,TotalGiving,DONOR_IND,BIRTH_DATE
0,1,23187.0,NaN,Married,Female,N,N,N,N,NaN,NaN,HOME,N,1,$0,$0,$0,$0,$0,$0,10.0,Y,NaN
1,2,77643.0,33.0,NaN,Female,N,Y,N,Y,NaN,UB,NaN,Y,0,$0,$0,$0,$0,$0,$0,2100.0,Y,1984-06-16
2,3,NaN,NaN,Married,Female,N,N,N,N,NaN,NaN,HOME,N,1,$0,$0,$0,$0,$0,$200,200.0,Y,NaN
3,4,47141.0,31.0,NaN,Female,N,Y,N,Y,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,0.0,N,1986-12-03
4,5,92555.0,68.0,NaN,Female,N,N,N,N,NaN,NaN,HOME,Y,0,$0,$0,$0,$0,$0,$0,505.0,Y,1949-09-11


In [207]:
donors['WEALTH_RATING'].unique()

array([nan, '$50,000-$99,999', '$100,000-$249,999', '$25,000-$49,999',
       '$250,000-$499,999', '$1-$24,999', '$1,000,000-$2,499,999',
       '$500,000-$999,999', '$2,500,000-$4,999,999'], dtype=object)

## 2.1) Data cleaning - donors 

In [208]:
# Include necessary rows only 

donors_cleaned = donors[["ID", "AGE", "MARITAL_STATUS", "GENDER", "ALUMNUS_IND", "WEALTH_RATING", "CON_YEARS", 
    "PrevFYGiving", "PrevFY1Giving", "PrevFY2Giving",
    "PrevFY3Giving", "PrevFY4Giving", "CurrFYGiving", "TotalGiving"
]].copy()

In [209]:
# Make Age group 

def age_group(age):
    try:
        age = float(age)
        if age < 20:
            return "Under 20"
        elif 20 <= age < 40:
            return "20-39"
        elif 40 <= age < 60:
            return "40-59"
        elif age >= 60:  
            return "60+"
        else: 
            return "Unknown"  
    except:
        return "Unknown"

In [210]:
# Apply age group function 

donors_cleaned['Age_group'] = donors_cleaned['AGE'].apply(age_group) 

In [211]:
donors_cleaned.head(5)

,ID,AGE,MARITAL_STATUS,GENDER,ALUMNUS_IND,WEALTH_RATING,CON_YEARS,PrevFYGiving,PrevFY1Giving,PrevFY2Giving,PrevFY3Giving,PrevFY4Giving,CurrFYGiving,TotalGiving,Age_group
0,1,NaN,Married,Female,N,NaN,1,$0,$0,$0,$0,$0,$0,10.0,Unknown
1,2,33.0,NaN,Female,Y,NaN,0,$0,$0,$0,$0,$0,$0,2100.0,20-39
2,3,NaN,Married,Female,N,NaN,1,$0,$0,$0,$0,$0,$200,200.0,Unknown
3,4,31.0,NaN,Female,Y,NaN,0,$0,$0,$0,$0,$0,$0,0.0,20-39
4,5,68.0,NaN,Female,N,NaN,0,$0,$0,$0,$0,$0,$0,505.0,60+


In [212]:
# Remove $ from donation amount columns 

giving_cols = ["PrevFYGiving", "PrevFY1Giving", "PrevFY2Giving", "PrevFY3Giving", "PrevFY4Giving", "CurrFYGiving"]
for col in giving_cols:
    donors_cleaned[col] = donors_cleaned[col].replace('[\$,]', '', regex=True).astype(float)


<>:5: SyntaxWarning: invalid escape sequence '\$'
<>:5: SyntaxWarning: invalid escape sequence '\$'
C:\Users\lg2021\AppData\Local\Temp\ipykernel_12372\1094359290.py:5: SyntaxWarning: invalid escape sequence '\$'
  donors_cleaned[col] = donors_cleaned[col].replace('[\$,]', '', regex=True).astype(float)


In [213]:
# Remove NA from Total giving column 

donors_cleaned["TotalGiving"] = donors_cleaned["TotalGiving"].fillna(0).astype(float)

In [214]:
donors_cleaned.head(10)

,ID,AGE,MARITAL_STATUS,GENDER,ALUMNUS_IND,WEALTH_RATING,CON_YEARS,PrevFYGiving,PrevFY1Giving,PrevFY2Giving,PrevFY3Giving,PrevFY4Giving,CurrFYGiving,TotalGiving,Age_group
0,1,NaN,Married,Female,N,NaN,1,0.0,0.0,0.0,0.0,0.0,0.0,10.0,Unknown
1,2,33.0,NaN,Female,Y,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,2100.0,20-39
2,3,NaN,Married,Female,N,NaN,1,0.0,0.0,0.0,0.0,0.0,200.0,200.0,Unknown
3,4,31.0,NaN,Female,Y,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20-39
4,5,68.0,NaN,Female,N,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,505.0,60+
5,6,57.0,NaN,Male,N,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40-59
6,7,NaN,NaN,Male,N,NaN,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Unknown
7,8,NaN,Married,Female,N,NaN,1,0.0,0.0,0.0,0.0,0.0,0.0,170.0,Unknown
8,9,NaN,Single,Uknown,N,NaN,0,5.0,0.0,0.0,0.0,0.0,0.0,5.0,Unknown
9,10,NaN,Married,Female,N,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,Unknown


## 2.2) Data cleaning - contacts 

In [215]:
# Contacts data cleaning

contacts.head()

,Staff Name,Method,Date,Summary,Substantive,Donor,Outcome
0,Ann Marie Levier,Email,12/1/17,Sent Dominic an email asking for an update if ...,Y,Dominic Richards,Positive
1,Ann Marie Levier,Email,12/1/17,"Emailed Dr. Jonathan Winters, who have ties to...",Y,William Bendrich,Positive
2,Ann Marie Levier,Letter,11/30/17,"Sent Belinda an upgrade brochure, appeal lette...",Y,Belinda Carlyle,Positive
3,Deborah Mettier,Phone,11/29/17,$50k to extend Kendrick Memorial Undergraduate...,Y,Donna Gilbert,Positive
4,Ann Marie Levier,Letter,11/27/17,Sent upgrade brochure asking Jeffrey and Julie...,Y,Jeffrey Jantz,Positive


In [216]:
contacts_cleaned = contacts[["Staff Name", "Donor", "Method", "Date", "Substantive", "Outcome"]].copy()

In [217]:
# Convert Date column to datetime type 

contacts_cleaned['Date'] = pd.to_datetime(contacts_cleaned['Date'], format ="%m/%d/%y")                                      

In [218]:
contacts_cleaned.head()

,Staff Name,Donor,Method,Date,Substantive,Outcome
0,Ann Marie Levier,Dominic Richards,Email,2017-12-01,Y,Positive
1,Ann Marie Levier,William Bendrich,Email,2017-12-01,Y,Positive
2,Ann Marie Levier,Belinda Carlyle,Letter,2017-11-30,Y,Positive
3,Deborah Mettier,Donna Gilbert,Phone,2017-11-29,Y,Positive
4,Ann Marie Levier,Jeffrey Jantz,Letter,2017-11-27,Y,Positive


## 3.1) Connect to SQLite DB

In [219]:
# Connect to SQLite DB 

connection = sqlite3.connect("nonprofit_operations.db") 

In [220]:
donors_cleaned.to_sql("donors", connection, if_exists="replace", index=False) 

34508

In [221]:
contacts_cleaned.to_sql("contacts", connection, if_exists="replace", index=False) 

196

## 3.2) Connect to SQLite, Calculate KPIs 

In [222]:
# KPI 1 : Average donation amount by Age group 

query1 = """
SELECT 
    Age_group
    ,ROUND(AVG(TotalGiving), 2) AS Avg_giving 
    ,CASE 
        WHEN Age_group = 'Under 20' THEN 1
        WHEN Age_group = '20-39' THEN 2
        WHEN Age_group = '40-59' THEN 3
        WHEN Age_group = '60' THEN 4
        WHEN Age_group = 'Unknown' THEN 5
    END AS order_col 
FROM donors 
GROUP BY Age_group 
ORDER BY order_col 
    
""" 

In [223]:
# KPI 2: Average donation amount by Alumni 

query2 = """ 
SELECT 
    ALUMNUS_IND 
    ,COUNT(DISTINCT ID) AS Donor_Count 
    ,ROUND(AVG(TotalGiving), 2) AS Avg_giving 
FROM donors 
GROUP BY ALUMNUS_IND 
ORDER BY ALUMNUS_IND ASC 
"""

In [224]:
# KPI 3: Donation amount by Wealth rating 

query3 = """
SELECT 
    WEALTH_RATING,
    COUNT(*) AS donor_count,
    ROUND(AVG(TotalGiving), 2) AS Avg_giving
FROM donors
WHERE WEALTH_RATING IS NOT NULL
GROUP BY WEALTH_RATING
ORDER BY Avg_giving DESC;
""" 

In [225]:
# KPI 4: Average giving amount by loyalty 

query4 = """ 
SELECT 
    COUNT(*) As donor_count
    ,ROUND(AVG(TotalGiving), 2) As Avg_giving 
FROM donors 
WHERE CON_YEARS >=3; 
"""

In [226]:
# KPI 5: Average donation pattern in the past 5 years 

query5 = """
SELECT 
    ROUND(AVG(PrevFYGiving), 2) as Avg_prev 
    ,ROUND(AVG(PrevFY1Giving), 2) as Avg_prev1 
    ,ROUND(AVG(PrevFY2Giving), 2) as Avg_prev2 
    ,ROUND(AVG(PrevFY3Giving), 2) as Avg_prev3 
    ,ROUND(AVG(PrevFY4Giving), 2) as Avg_prev4 
    ,ROUND(AVG(CurrFYGiving),2) as Avg_curr 
FROM donors; 
"""

In [227]:
# KPI 6: Number of contacts by Methods 

query6 = """ 
SELECT 
    Method 
    ,count(*) as Contact_count 
FROM contacts 
GROUP BY Method 
ORDER BY Contact_count DESC; 
"""

In [228]:
# KPI 7: Number of Contacts by Staff 

query7 = """ 
SELECT 
    [Staff Name] 
    ,count(*) as Contact_count 
FROM contacts 
GROUP BY [Staff Name]  
ORDER BY Contact_count DESC; 
""" 

In [229]:
# KPI 8: Number of contacts by Substance 

query8 = """ 
SELECT 
    COUNT(*) AS total_contacts,
    SUM(CASE WHEN Substantive = 'Y' THEN 1 ELSE 0 END) AS substantive_contacts,
    ROUND(100.0 * SUM(CASE WHEN Substantive = 'Y' THEN 1 ELSE 0 END) / COUNT(*), 2) AS percent_substantive
FROM contacts;
""" 

In [230]:
# KPI 9: Number of contacts by Month 

query9 = """ 
SELECT 
    strftime('%Y-%m', Date) AS contact_month,
    COUNT(*) AS contact_count
FROM contacts
GROUP BY contact_month
ORDER BY contact_month;
 
"""

In [231]:
contacts_cleaned['Date'].unique()

<DatetimeArray>
['2017-12-01 00:00:00', '2017-11-30 00:00:00', '2017-11-29 00:00:00',
 '2017-11-27 00:00:00', '2017-11-24 00:00:00', '2017-11-20 00:00:00',
 '2017-10-30 00:00:00', '2017-10-27 00:00:00', '2017-10-26 00:00:00',
 '2017-10-25 00:00:00', '2017-10-18 00:00:00', '2017-10-03 00:00:00',
 '2017-10-01 00:00:00', '2017-09-27 00:00:00', '2017-09-24 00:00:00',
 '2017-09-22 00:00:00', '2017-09-15 00:00:00', '2017-08-31 00:00:00',
 '2017-08-30 00:00:00', '2017-08-11 00:00:00', '2017-08-07 00:00:00',
 '2017-07-29 00:00:00', '2017-07-27 00:00:00', '2017-07-18 00:00:00',
 '2017-07-15 00:00:00', '2017-07-13 00:00:00', '2017-07-12 00:00:00',
 '2017-07-11 00:00:00', '2017-07-10 00:00:00', '2017-07-09 00:00:00',
 '2017-07-08 00:00:00', '2017-07-07 00:00:00', '2017-08-01 00:00:00',
 '2017-08-02 00:00:00', '2017-08-27 00:00:00']
Length: 35, dtype: datetime64[ns]

## 3.3) Save to DataFrames 

In [232]:
# Save query to dataframe 

df1 = pd.read_sql_query(query1, connection) 

df1 = df1.drop(columns = 'order_col')
df1 = df1.sort_values("Age_group", key=lambda x: x.map({
    "Under 20": 1, "20-39": 2, "40-59": 3, "60": 4, "Unknown": 5
}))
print(df1) 

  Age_group  Avg_giving
1  Under 20     6037.39
2     20-39      729.29
3     40-59     2293.84
4   Unknown     1618.41
0       60+    10255.91


In [233]:
df2 = pd.read_sql_query(query2, connection) 
print(df2)

  ALUMNUS_IND  Donor_Count  Avg_giving
0           N        26086     1384.99
1           Y         8422     5394.44


In [234]:
df3 = pd.read_sql_query(query3, connection) 
print(df3)

           WEALTH_RATING  donor_count  Avg_giving
0      $500,000-$999,999           81    42524.90
1             $1-$24,999          580    18506.15
2  $1,000,000-$2,499,999           59     2936.24
3      $100,000-$249,999          511     2006.49
4      $250,000-$499,999          265      655.30
5        $50,000-$99,999          645      398.59
6        $25,000-$49,999          564      397.79
7  $2,500,000-$4,999,999            4      106.25


In [235]:
df4 = pd.read_sql_query(query4, connection) 
print(df4)

   donor_count  Avg_giving
0         4066    12803.39


In [236]:
df5 = pd.read_sql_query(query5, connection) 
print(df5)

   Avg_prev  Avg_prev1  Avg_prev2  Avg_prev3  Avg_prev4  Avg_curr
0    377.62       96.2      63.75      57.12     126.63    197.79


In [237]:
df6 = pd.read_sql_query(query6, connection) 
print(df6)

   Method  Contact_count
0   Visit             82
1   Email             72
2   Phone             36
3  Letter              6


In [238]:
df7 = pd.read_sql_query(query7, connection) 
print(df7)

         Staff Name  Contact_count
0    Rashi Mohinder             68
1  Ann Marie Levier             42
2      April Catson             32
3   Deborah Mettier             16
4       Bill Bamers             12
5    Carlos Bendiga             10
6        Tos Norani              8
7       Joan Joffey              8


In [239]:
df8 = pd.read_sql_query(query8, connection) 
print(df8)

   total_contacts  substantive_contacts  percent_substantive
0             196                    53                27.04


In [240]:
df9 = pd.read_sql_query(query9, connection) 
print(df9)

  contact_month  contact_count
0       2017-07             80
1       2017-08             86
2       2017-09             10
3       2017-10             12
4       2017-11              6
5       2017-12              2


## 3.4) Close SQLite connection 

In [241]:
connection.close 

<function Connection.close()>

## 4.1) Automation: Save to Excel File 

In [242]:
with pd.ExcelWriter("monthly_kpi_report.xlsx") as writer:
    df1.to_excel(writer, sheet_name="Age Group", index=False)
    df2.to_excel(writer, sheet_name="Alumnus", index=False)
    df3.to_excel(writer, sheet_name="Wealth", index=False)
    df4.to_excel(writer, sheet_name="Loyal Donors", index=False)
    df5.to_excel(writer, sheet_name="Trends for 5 Years", index=False)
    df6.to_excel(writer, sheet_name="Contact count by Methods", index=False)
    df7.to_excel(writer, sheet_name="Contact count by Staff", index=False)
    df8.to_excel(writer, sheet_name="Substantive contact count", index=False)
    df9.to_excel(writer, sheet_name="Contact count by Month", index=False)


In [243]:
# Save to Excel file 

donors_cleaned.to_excel("donors_cleaned.xlsx", index=False) 

In [244]:
contacts_cleaned.to_excel("contacts_cleaned.xlsx", index=False) 

## 4.2) Automation: Send to email 

In [245]:
import yagmail

sender_email = "rachelhyesoo@gmail.com"
app_password = "qzcz piyp dhoa vhew"

receiver_email = "rachelhyesoo@gmail.com"

yag = yagmail.SMTP(sender_email, app_password)

yag.send(
    to=receiver_email,
    subject="Monthly KPI Report",
    
    contents="""
    
    Please find attached a monthly KPI Report for non profit organisation.
    
    """,
    
    attachments="monthly_kpi_report.xlsx"
)

print("Report Sent!") 


Report Sent!
